# Redes Convolucionales Modernas

In [1]:
%matplotlib inline
import torch
import torchvision
from torch import nn
import matplotlib.pyplot as plt
from torchvision.io import read_image
from torchvision.io import decode_image
import torchvision.transforms as T
from torch.nn import functional as F
from torch.utils import data
import numpy as np

def init_cnn(module):
    if type(module) == nn.Linear or type(module) == nn.Conv2d:
        nn.init.xavier_uniform_(module.weight)


In [2]:
def load_data_fashion_mnist(batch_size, resize=None):
    trans = [T.ToTensor()]
    if resize:
        trans.insert(0, T.Resize(resize))
    trans = T.Compose(trans)
    mnist_train = torchvision.datasets.FashionMNIST(
        root="../data", train=True, transform=trans, download=True)
    mnist_test = torchvision.datasets.FashionMNIST(
        root="../data", train=False, transform=trans, download=True)
    return (data.DataLoader(mnist_train, batch_size, shuffle=True,
                            num_workers=1),
            data.DataLoader(mnist_test, batch_size, shuffle=False,
                            num_workers=1))

def accuracy(y_hat, y):
    """Compute the number of correct predictions."""
    if len(y_hat.shape) > 1 and y_hat.shape[1] > 1:
        y_hat = y_hat.argmax(axis=1)
    cmp = y_hat.type(y.dtype) == y
    return float(cmp.type(y.dtype).sum())


In [3]:
def train_FashionMNIST_classifier(model, lr, num_epochs, resize=None):
  batch_size= 128
  device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
  model = model.to(device)
  loss = nn.CrossEntropyLoss(reduction='none')
  trainer = torch.optim.Adam(model.parameters())
  train_iter, test_iter = load_data_fashion_mnist(batch_size,resize=resize)

  for epoch in range(num_epochs):
      L = 0.0
      N = 0
      Acc = 0.0
      TestAcc = 0.0
      TestN = 0
      for X, y in train_iter:
          X, y = X.to(device), y.to(device)
          l = loss(model(X),y)
          trainer.zero_grad()
          l.mean().backward()
          trainer.step()
          L += l.sum()
          N += l.numel()
          Acc += accuracy(model(X), y)
      for X, y in test_iter:
          X, y = X.to(device), y.to(device)
          TestN += y.numel()
          TestAcc += accuracy(model(X), y)
      print(f'epoch {epoch + 1}, loss {(L/N):f}\
            , train accuracy  {(Acc/N):f}, test accuracy {(TestAcc/TestN):f}')

# Redes que usan bloques (VGG)



Si bien AlexNet ofreció evidencia empírica de que las CNN profundas pueden lograr buenos resultados, no proporcionó una plantilla general para guiar a los investigadores posteriores en el diseño de nuevas redes. En las siguientes secciones, presentaremos varios conceptos heurísticos comúnmente utilizados para diseñar redes profundas.

El diseño de las arquitecturas de redes neuronales se ha vuelto cada vez más abstracto, y los investigadores han pasado de pensar en términos de neuronas individuales a capas completas, y ahora a bloques, patrones repetitivos de capas.


La idea de usar bloques surgió por primera vez del Visual Geometry Group (VGG) de la Universidad de Oxford, en su red VGG del mismo nombre. Es fácil implementar estas estructuras repetidas en código con cualquier framework moderno de Deep Learning mediante el uso de bucles y subrutinas.

### **Bloques VGG**

El componente básico de las CNN es una secuencia de lo siguiente:
* (i) una capa convolucional con padding para mantener la resolución,
* (ii) una no linealidad como ReLU,
* (iii) una capa de pooling como max-pooling para reducir la resolución.

Uno de los problemas con este enfoque es que la resolución espacial disminuye con bastante rapidez. En particular, esto impone un límite estricto de $\log_2 d$ capas convolucionales en la red antes de que se agoten todas las dimensiones ($d$). Por ejemplo, en el caso de ImageNet, sería imposible tener más de 8 capas convolucionales de esta forma.

La idea clave de VGG era un bloque que utilizara múltiples convoluciones antes de reducir la dimensionalidad a través de max-pooling. Estaban interesados ​​principalmente en determinar si las redes profundas o las amplias funcionan mejor. Por ejemplo, la aplicación sucesiva de dos convoluciones de $3 \times 3$ toca los mismos píxeles que una sola convolución de $5 \times 5$. Al mismo tiempo, este último usa aproximadamente tantos parámetros ($25 \cdot c^2$) como tres convoluciones $3 \times 3$ ($3 \cdot 9 \cdot c^2$).

En un análisis bastante detallado, demostraron que las redes profundas y estrechas superan significativamente a sus contrapartes superficiales. Esto puso al aprendizaje profundo en la búsqueda de redes cada vez más profundas con más de 100 capas para aplicaciones típicas. Apilar convoluciones de $3 \times 3$ se había convertido en un estándar de oro en las redes profundas posteriores.

Volviendo a VGG: un bloque VGG consta de una *secuencia* de convoluciones con kernels de $3\times3$ con padding de 1 (manteniendo la altura y el ancho) seguida de una capa de max-pooling de $2\times 2$ con stride de 2 (reduciendo a la mitad la altura y ancho después de cada bloque). En el siguiente código, definimos una función llamada `vgg_block` para implementar un bloque VGG.

La siguiente función toma dos argumentos, correspondientes al número de capas convolucionales `num_convs` y al número de canales de salida `num_channels`.




In [4]:
def vgg_block(num_convs, out_channels):
    layers = []
    for _ in range(num_convs):
        layers.append(nn.LazyConv2d(out_channels, kernel_size=3, padding=1))
        layers.append(nn.ReLU())
    layers.append(nn.MaxPool2d(kernel_size=2,stride=2))
    return nn.Sequential(*layers)

### Red VGG



Al igual que AlexNet y LeNet, la red VGG se puede dividir en dos partes: la primera consiste principalmente en capas convolucionales y de pooling y la segunda consiste en capas densas que son idénticas a las de AlexNet. La diferencia clave es que las capas convolucionales se agrupan en transformaciones no lineales que dejan la dimensionalidad sin cambios, seguido de un paso de reducción de resolución, como se muestra en la figura.

![Imgur](https://i.imgur.com/OnQUYgM.png)

De AlexNet a VGG. La diferencia clave es que VGG consta de bloques de capas, mientras que las capas de AlexNet están diseñadas individualmente.

La parte convolucional de la red conecta varios bloques VGG en sucesión. Esta agrupación de convoluciones es un patrón que se ha mantenido casi sin cambios durante la última década, aunque la elección específica de operaciones ha sufrido modificaciones considerables. La variable conv_arch consiste en una lista de tuplas (una por bloque), donde cada una contiene dos valores: el número de capas convolucionales y el número de canales de salida, que son precisamente los argumentos necesarios para llamar a la función vgg_block. Como tal, VGG define una familia de redes en lugar de solo una manifestación específica. Para construir una red específica, simplemente iteramos sobre arch para componer los bloques.

In [5]:
class VGG(nn.Module):
    def __init__(self, arch, lr=0.1, num_classes=10):
        super().__init__()
        conv_blks = []
        for (num_convs, out_channels) in arch:
            conv_blks.append(vgg_block(num_convs, out_channels))
        self.net = nn.Sequential(
            *conv_blks, nn.Flatten(),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(4096), nn.ReLU(), nn.Dropout(0.5),
            nn.LazyLinear(num_classes))
        self.net.apply(init_cnn)

    def forward(self, X):
        return self.net(X)

La red VGG original tenía 5 bloques convolucionales, entre los cuales los dos primeros tienen una capa convolucional cada uno y los últimos tres contienen dos capas convolucionales cada uno. El primer bloque tiene 64 canales de salida y cada bloque subsiguiente duplica la cantidad de canales de salida, hasta que ese número llega a 512. Dado que esta red usa 8 capas convolucionales y 3 capas densas, a menudo se la denomina VGG-11.


In [6]:
# https://arxiv.org/abs/1409.1556
# Very Deep Convolutional Networks for Large-Scale Image Recognition
# Karen Simonyan, Andrew Zisserman

model_vgg = VGG(arch=((1, 64), (1, 128), (2, 256), (2, 512), (2, 512)))
print(model_vgg)

VGG(
  (net): Sequential(
    (0): Sequential(
      (0): LazyConv2d(0, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (1): Sequential(
      (0): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (2): Sequential(
      (0): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): LazyConv2d(0, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    )
    (3): Sequential(
      (0): LazyConv2d(0, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): ReLU()
      (2): LazyConv2d(0, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): MaxPool2d(

Como puede ver, redujimos a la mitad el alto y el ancho de cada bloque, alcanzando finalmente un alto y un ancho de 7 antes de aplanar las representaciones para que las capas densas de la red las procesen. El paper de VGG describió varias otras variantes de la red. De hecho, se ha convertido en la norma proponer familias de redes con diferentes compromisos entre velocidad y precisión al introducir una nueva arquitectura.

### Entrenamiento

**Dado que VGG-11 es computacionalmente más exigente que AlexNet, construimos una red con una menor cantidad de canales.**
Esto es más que suficiente para el entrenamiento con Fashion-MNIST.
Nuevamente, observe la estrecha coincidencia entre la validación y la pérdida de entrenamiento, lo que sugiere solo una pequeña cantidad de sobreajuste.


In [7]:
# con T4 demora menos de 2 minutos
lr, num_epochs = 0.1, 1
model_vgg_small = VGG(arch=((1, 16), (1, 32), (2, 64), (2, 128), (2, 128)), lr=0.01)
train_FashionMNIST_classifier(model_vgg_small,lr,num_epochs,resize=(224, 224))

100%|██████████| 26.4M/26.4M [00:02<00:00, 8.92MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 210kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.93MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 15.3MB/s]


epoch 1, loss 0.540215            , train accuracy  0.809217, test accuracy 0.873700


# Redes sin Capas Densas (NiN)



LeNet, AlexNet y VGG comparten un patrón de diseño común: extraer características que explotan la estructura espacial a través de una secuencia de convoluciones y capas de pooling y postprocesar las representaciones a través de capas densas. Las mejoras sobre LeNet por parte de AlexNet y VGG radican principalmente en cómo estas redes posteriores amplían y profundizan estos dos módulos.

![Imgur](https://i.imgur.com/3pcQyXf.png)

Este diseño plantea dos grandes desafíos:
1. Las capas densas al final de la arquitectura consumen una **gran cantidad de parámetros**. Por ejemplo, incluso un modelo simple como VGG-11 requiere una matriz monstruosa de 25088 × 4096, ocupando casi 400 MB de RAM en precisión simple (float32). Este es un impedimento significativo para la computación, en particular en dispositivos móviles y embebidos. Después de todo, incluso los teléfonos móviles actuales de gama alta no tienen más de 8 GB de RAM. En el momento en que se inventó VGG, esto era un orden de magnitud menor (el iPhone 4S tenía 512 MB). Como tal, hubiera sido difícil justificar gastar la mayor parte de la memoria en un clasificador de imágenes.

2. Es igualmente **imposible agregar capas densas antes en la red** para aumentar el grado de no linealidad: hacerlo destruiría la estructura espacial y requeriría potencialmente incluso más memoria.

Los bloques de Network in Network (NiN) ofrecen una alternativa, capaz de resolver ambos problemas en una estrategia simple. Se propusieron en base a una idea muy simple:
1. usar convoluciones 1×1 para agregar no linealidades locales en las activaciones del canal y
2. usar global average pooling para resumir la información a través de todas las ubicaciones en la última capa de representación.

### Bloques NiN


Recuerde la clase anterior donde discutimos que las entradas y salidas de las capas convolucionales consisten en tensores de cuatro dimensiones con ejes correspondientes al ejemplo, canal, altura y ancho. Recuerde también que las entradas y salidas de las capas densas suelen ser tensores bidimensionales correspondientes al ejemplo y la característica.



```
convolution_input.shape = (batch_size, channels, heigth, width)

linear_input.shape = (batch_size, feature_dim)
```



La idea detrás de NiN es aplicar una capa densa en cada ubicación de píxel (para cada alto y ancho). La convolución de $1\times1$ se puede pensar como una capa densa que actúa de forma independiente sobre los canales en cada ubicación de píxel.

![Imgur](https://i.imgur.com/XyGXPzQ.png)



La siguiente figura ilustra las principales diferencias estructurales entre VGG y NiN, y sus bloques. Tenga en cuenta tanto la diferencia en los bloques de NiN (la convolución inicial es seguida por convoluciones $1\times1$, mientras que VGG retiene convoluciones $3\times3$) y al final donde ya no necesitamos una capa densa gigante.

![Imgur](https://i.imgur.com/QOw5mml.png)

In [8]:
# https://iq.opengenus.org/global-average-pooling/

def nin_block(out_channels, kernel_size, strides, padding):
    return nn.Sequential(
        nn.LazyConv2d(out_channels, kernel_size, strides, padding), nn.ReLU(),
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU(),
        nn.LazyConv2d(out_channels, kernel_size=1), nn.ReLU())

## Modelo NiN

NiN usa los mismos tamaños de convolución iniciales que AlexNet (se propuso poco después). Los tamaños de kernel son $11\times11$, $5\times5$ y $3\times3$, respectivamente, y la cantidad de canales de salida coincide con la de AlexNet. A cada bloque de NiN le sigue una capa de max-pooling con un stride de 2 y un tamaño de ventana de $3\times3$.

La segunda diferencia significativa entre NiN y tanto AlexNet como VGG es que NiN evita por completo las capas densas. En su lugar, NiN utiliza un bloque de NiN con una cantidad de canales de salida igual a la cantidad de clases de etiquetas, seguido de una capa de average pooling global, lo que produce un vector de logits. Este diseño reduce significativamente la cantidad de parámetros de modelo requeridos, aunque a expensas de un aumento potencial en el tiempo de entrenamiento.


In [9]:
class NiN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nin_block(96, kernel_size=11, strides=4, padding=0),
            nn.MaxPool2d(3, stride=2),
            nin_block(256, kernel_size=5, strides=1, padding=2),
            nn.MaxPool2d(3, stride=2),
            nin_block(384, kernel_size=3, strides=1, padding=1),
            nn.MaxPool2d(3, stride=2),
            nn.Dropout(0.5),
            nin_block(num_classes, kernel_size=3, strides=1, padding=1),
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten())
        self.net.apply(init_cnn)

    def forward(self, X):
        return self.net(X)


Creamos un ejemplo de datos para ver la forma de la salida de cada bloque.


In [10]:
model = NiN()
X = torch.randn(1, 1, 224, 224)
for layer in model.net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

Sequential output shape:	 torch.Size([1, 96, 54, 54])
MaxPool2d output shape:	 torch.Size([1, 96, 26, 26])
Sequential output shape:	 torch.Size([1, 256, 26, 26])
MaxPool2d output shape:	 torch.Size([1, 256, 12, 12])
Sequential output shape:	 torch.Size([1, 384, 12, 12])
MaxPool2d output shape:	 torch.Size([1, 384, 5, 5])
Dropout output shape:	 torch.Size([1, 384, 5, 5])
Sequential output shape:	 torch.Size([1, 10, 5, 5])
AdaptiveAvgPool2d output shape:	 torch.Size([1, 10, 1, 1])
Flatten output shape:	 torch.Size([1, 10])


## Entrenamiento

Como antes, usamos Fashion-MNIST para entrenar al modelo usando el mismo
optimizador que usamos para AlexNet y VGG.


In [11]:
# con T4 demora 7.5 minutos
lr, num_epochs = 0.2, 5
nin = NiN()
train_FashionMNIST_classifier(nin,lr,num_epochs,resize=(224, 224))

epoch 1, loss 1.858431            , train accuracy  0.330450, test accuracy 0.493700
epoch 2, loss 1.303491            , train accuracy  0.562483, test accuracy 0.573400
epoch 3, loss 1.210438            , train accuracy  0.591167, test accuracy 0.575500
epoch 4, loss 1.151125            , train accuracy  0.600217, test accuracy 0.600200
epoch 5, loss 1.109099            , train accuracy  0.606933, test accuracy 0.604600


# Redes con Múltiples Ramas (GoogLeNet)




En 2014, GoogLeNet ganó el ImageNet Challenge  usando una estructura que combinaba las fortalezas de NiN , con la repetición de bloques de VGG, y un cóctel de diferentes kernels de convolución.

Podría decirse que también es la primera red que exhibe una clara distinción entre la base (ingreso de datos), el cuerpo (procesamiento de datos) y la cabeza (predicción) en una CNN. Este patrón de diseño ha persistido desde entonces en el diseño de redes profundas:
* La **base** está dada por las primeras 2-3 convoluciones que operan en la imagen. Extraen características de bajo nivel de las imágenes subyacentes.
* Esto es seguido por un **cuerpo** de bloques convolucionales.
* Finalmente, la **cabeza** asigna las características obtenidas hasta el momento al problema requerido de clasificación, segmentación, detección o seguimiento en cuestión.

La contribución clave en GoogLeNet fue el diseño del cuerpo de la red. Resolvió el problema de seleccionar núcleos de convolución de una manera ingeniosa. Mientras que otros trabajos intentaron identificar qué convolución, que oscilaba entre  $1 \times 1$ y $11 \times 11$, sería la mejor, GoogLeNet simplemente concatenó múltiples ramas de convoluciones. A continuación, presentamos una versión ligeramente simplificada de GoogLeNet: el diseño original incluía una serie de trucos para estabilizar el entrenamiento a través de funciones de pérdida intermedia, aplicadas a múltiples capas de la red. Ya no son necesarios debido a la disponibilidad de algoritmos de entrenamiento mejorados.

## Bloques Inception

El bloque convolucional básico en GoogLeNet se llama bloque Inception, derivado del meme "we need to go deeper" de la película Inception.

![Imgur](https://i.imgur.com/CZNwqzw.png)

Como se muestra en la siguiente figura, el bloque de inicio consta de cuatro ramas paralelas. Las primeras tres ramas usan capas convolucionales con tamaños de ventana de $1\times 1$, $3\times 3$, y $5\times 5$ para extraer información de diferentes tamaños espaciales. Las dos ramas del medio también agregan una convolución de $1\times 1$ de la entrada para reducir la cantidad de canales, lo que reduce la complejidad del modelo. La cuarta rama utiliza una capa de agrupación máxima de $3\times 3$, seguida de una capa convolucional de $1\times 1$ para cambiar la cantidad de canales.

![Imgur](https://i.imgur.com/QfJzT2S.png)

Las cuatro ramas utilizan el la cantidad adecuada de padding para dar a la entrada y a la salida la misma altura y anchura. Finalmente, las salidas a lo largo de cada rama se concatenan a lo largo de la dimensión del canal y comprenden la salida del bloque. Los hiperparámetros que se pueden "tunear" del bloque Inception son el número de canales de salida por capa.

In [12]:
class Inception(nn.Module):
    # `c1`--`c4` son el número de canales de salida para cada rama
    def __init__(self, c1, c2, c3, c4, **kwargs):
        super(Inception, self).__init__(**kwargs)
        # Branch 1
        self.b1_1 = nn.LazyConv2d(c1, kernel_size=1)
        # Branch 2
        self.b2_1 = nn.LazyConv2d(c2[0], kernel_size=1)
        self.b2_2 = nn.LazyConv2d(c2[1], kernel_size=3, padding=1)
        # Branch 3
        self.b3_1 = nn.LazyConv2d(c3[0], kernel_size=1)
        self.b3_2 = nn.LazyConv2d(c3[1], kernel_size=5, padding=2)
        # Branch 4
        self.b4_1 = nn.MaxPool2d(kernel_size=3, stride=1, padding=1)
        self.b4_2 = nn.LazyConv2d(c4, kernel_size=1)

    def forward(self, x):
        b1 = F.relu(self.b1_1(x))
        b2 = F.relu(self.b2_2(F.relu(self.b2_1(x))))
        b3 = F.relu(self.b3_2(F.relu(self.b3_1(x))))
        b4 = F.relu(self.b4_2(self.b4_1(x)))
        return torch.cat((b1, b2, b3, b4), dim=1)

Para ganar algo de intuición de por qué esta red funciona tan bien, considere la combinación de los filtros. Exploran la imagen en una variedad de tamaños de filtro. Esto significa que los detalles en diferentes extensiones se pueden reconocer de manera eficiente mediante filtros de diferentes tamaños. Al mismo tiempo, podemos asignar diferentes cantidades de parámetros para diferentes filtros.

## **Modelo de GoogLeNet**

Como se muestra en la figura, GoogLeNet usa una pila de un total de 9 bloques inception, organizados en 3 grupos con un max-pooling en el medio, y el average pooling global en su cabeza para generar sus estimaciones.

![Imgur](https://i.imgur.com/4lEBKNO.png)

En su base, el primer bloque es similar a AlexNet y LeNet.

Ahora podemos implementar GoogLeNet pieza por pieza. Comencemos con la base. El primer bloque utiliza una capa convolucional $7\times 7$ de 64 canales.


In [13]:
class B1(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
            nn.LazyConv2d(64, kernel_size=7, stride=2, padding=3),
            nn.ReLU(), nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    def forward(self, X):
        return self.net(X)

El segundo módulo utiliza dos capas convolucionales:
primero, una capa convolucional de $1\times 1$ de 64 canales, seguida de una capa convolucional de $3\times 3$ que triplica el número de canales.  En este punto tenemos 192 canales.


In [14]:
class B2(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(
          nn.LazyConv2d(64, kernel_size=1), nn.ReLU(),
          nn.LazyConv2d(192, kernel_size=3, padding=1), nn.ReLU(),
          nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    def forward(self, X):
        return self.net(X)


El tercer módulo conecta dos bloques Inception completos en serie.
El número de canales de salida del primer bloque Inception es
$64+128+32+32=256$. Esto equivale a una relación del número de canales de salida entre las cuatro ramas de $2:4:1:1$. Logrando esto, primero reducimos las dimensiones de entrada en $\frac{1}{2}$ y en $\frac{1}{12}$ en la segunda y tercera rama respectivamente para llegar a $96 = 192/2$ y $16 = 192/12$ canales respectivamente.

El número de canales de salida del segundo bloque Inception se incrementa a $128+192+96+64=480$, lo que arroja una proporción de $128:192:96:64 = 4:6:3:2$. Como antes, necesitamos reducir el número de dimensiones intermedias en el segundo y tercer canal. Una escala de $\frac{1}{2}$ y $\frac{1}{8}$ respectivamente es suficiente, produciendo canales de $128$ y $32$ respectivamente. Esto es capturado por los argumentos de los siguientes constructores de bloques `Inception`.


In [15]:
class B3(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(Inception(64, (96, 128), (16, 32), 32),
                         Inception(128, (128, 192), (32, 96), 64),
                         nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    def forward(self, X):
        return self.net(X)


El cuarto módulo es más complicado.
Conecta cinco bloques Inception en serie, y tienen $192+208+48+64=512$, $160+224+64+64=512$,
$128+256+64+64=512$, $112+288+64+64=528$, y $256+320+128+128=832$ canales de salida, respectivamente.
El número de canales asignados a estas sucursales es similar
a eso en el tercer módulo: la segunda rama con la capa convolucional $3\times 3$ genera la mayor cantidad de canales,
seguida por la primera rama con solo la capa convolucional $1\times 1$, la tercera rama con la capa convolucional $5\times 5$,
y la cuarta rama con la capa de max-pooling $3\times 3$.
Las ramas segunda y tercera reducirán primero el número de canales según la proporción.
Estas proporciones son ligeramente diferentes en diferentes bloques de Inception.


In [16]:
class B4(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(Inception(192, (96, 208), (16, 48), 64),
                         Inception(160, (112, 224), (24, 64), 64),
                         Inception(128, (128, 256), (24, 64), 64),
                         Inception(112, (144, 288), (32, 64), 64),
                         Inception(256, (160, 320), (32, 128), 128),
                         nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

    def forward(self, X):
        return self.net(X)


El quinto módulo tiene dos bloques Inception con $256+320+128+128=832$
y $384+384+128+128=1024$ canales de salida.
El número de canales asignados a cada rama es el mismo que en los módulos tercero y cuarto, pero difiere en valores específicos.
Cabe señalar que al quinto bloque le sigue la capa de salida.
Este bloque utiliza la capa de avg-pooling global para cambiar la altura y el ancho de cada canal a 1, al igual que en NiN. Finalmente, convertimos la salida en una matriz bidimensional seguida de una capa densa cuyo número de salidas es el número de clases de etiquetas.


In [17]:
class B5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.net = nn.Sequential(Inception(256, (160, 320), (32, 128), 128),
                         Inception(384, (192, 384), (48, 128), 128),
                         nn.AdaptiveAvgPool2d((1,1)), nn.Flatten())

    def forward(self, X):
        return self.net(X)

Now that we defined all blocks `b1` through `b5`, it's just a matter of assembling them all into a full network.


In [18]:
class GoogleNet(nn.Module):
  def __init__(self, num_classes=10):
    super(GoogleNet,self).__init__()
    self.b1 = B1()
    self.b2 = B2()
    self.b3 = B3()
    self.b4 = B4()
    self.b5 = B5()
    self.net = nn.Sequential(self.b1, self.b2, self.b3, self.b4,
                             self.b5, nn.LazyLinear(num_classes))
    self.net.apply(init_cnn)

  def forward(self, X):
    return self.net(X)

El modelo GoogLeNet es computacionalmente complejo. Tenga en cuenta la gran cantidad de hiperparámetros relativamente arbitrarios en términos de la cantidad de canales elegidos, la cantidad de bloques antes de la reducción de la dimensionalidad, la partición relativa de la capacidad entre canales, etc. Gran parte se debe al hecho de que en el momento en que GoogLeNet se introdujo, las herramientas automáticas para la definición de redes o la exploración de diseños aún no estaban disponibles. Por ejemplo, a estas alturas damos por sentado que un framework de deep learning competente es capaz de inferir automáticamente las dimensionalidades de los tensores de entrada. En ese momento, muchas de estas configuraciones tenían que ser especificadas explícitamente por el experimentador, lo que a menudo ralentizaba la experimentación activa. Además, las herramientas necesarias para la exploración automática todavía estaban cambiando y los experimentos iniciales consistían en gran medida en costosas exploraciones de fuerza bruta, algoritmos genéticos y estrategias similares.

Por ahora, la única modificación que realizaremos es reducir la altura y el ancho de entrada de 224 a 96 para tener un tiempo de entrenamiento razonable en Fashion-MNIST. Esto simplifica el cálculo. Echemos un vistazo a los cambios en la forma de la salida entre los distintos módulos.


In [19]:
g_net = GoogleNet()
print(g_net)

GoogleNet(
  (b1): B1(
    (net): Sequential(
      (0): LazyConv2d(0, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3))
      (1): ReLU()
      (2): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
  )
  (b2): B2(
    (net): Sequential(
      (0): LazyConv2d(0, 64, kernel_size=(1, 1), stride=(1, 1))
      (1): ReLU()
      (2): LazyConv2d(0, 192, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (3): ReLU()
      (4): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    )
  )
  (b3): B3(
    (net): Sequential(
      (0): Inception(
        (b1_1): LazyConv2d(0, 64, kernel_size=(1, 1), stride=(1, 1))
        (b2_1): LazyConv2d(0, 96, kernel_size=(1, 1), stride=(1, 1))
        (b2_2): LazyConv2d(0, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (b3_1): LazyConv2d(0, 16, kernel_size=(1, 1), stride=(1, 1))
        (b3_2): LazyConv2d(0, 32, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
        (b4_1

## Eficiencia Computacional



Una característica clave de GoogLeNet es que, en realidad, es más barato de calcular que sus predecesores y, al mismo tiempo, proporciona una mayor precisión. Esto marca el comienzo de un diseño de red mucho más deliberado que compensa el costo de entrenar una red con una reducción de errores. También marca el comienzo de la experimentación a nivel de bloque con hiperparámetros de diseño de red, aunque en ese momento era totalmente manual.

Dado que la cantidad de parámetros de una capa convolucional con kernel $(w_k,h_k)$, $ch_{in}$ canales de entrada y $ch_{out}$ canales de salida es:
$$w\times h \times (ch_{in}+1) \times ch_{out}$$ y la cantidad de operaciones necesarias para su cálculo es: $$(w_{in} \times h_{in} \times ch_{in}
 ) \times (w_{k} \times h_{k} \times ch_{out})$$
![Imgur](https://i.imgur.com/AM7eJH0.png)



In [20]:
total_canales = 256
input_channels = 192
#((shape of width of the filter * shape of height of the filter * number of filters in the previous layer+1)*number of filters)
param_inception = (input_channels+1)*64 +(input_channels+1)*96 +(input_channels+1)*16 + (input_channels+1)*32 + (3*3*(96+1))*128 + (5*5*(16+1))*32
param_3x3 = (input_channels+1)*3*3*total_canales
param_5x5 = (input_channels+1)*5*5*total_canales
#(shape_input*shape_kernel)
op_inception = 192*28*28*64*1*1 + 192*28*28*96*1*1 + 192*28*28*16*1*1 + 96*28*28*128*3*3 + 16*28*28*32*5*5+ 192*((28-3/1)+1)*((28-3/1)+1)*32*1*1
op_3x3 = input_channels*28*28*3*3*total_canales
op_5x5 = input_channels*28*28*5*5*total_canales
print("parámetros")
print((param_inception,param_3x3,param_5x5))
print("operaciones")
op_inception,op_3x3,op_5x5

parámetros
(165488, 444672, 1235200)
operaciones


(127385600.0, 346816512, 963379200)

## Entrenamiento

Como antes, entrenamos nuestro modelo utilizando el conjunto de datos Fashion-MNIST.
  Lo transformamos a una resolución de $96 \times 96$ pixel antes de invocar el procedimiento de entrenamiento.


In [21]:
# Con T4, 5 epochs demoran 5 min
lr, num_epochs = 0.01, 5
train_FashionMNIST_classifier(g_net,lr,num_epochs,resize=(96, 96))

epoch 1, loss 1.421684            , train accuracy  0.456700, test accuracy 0.743300
epoch 2, loss 0.513605            , train accuracy  0.824033, test accuracy 0.830400
epoch 3, loss 0.405730            , train accuracy  0.859867, test accuracy 0.848800
epoch 4, loss 0.348426            , train accuracy  0.881517, test accuracy 0.861000
epoch 5, loss 0.317027            , train accuracy  0.892600, test accuracy 0.887100
